# Aksara OCR — benchmark run (Colab Pro, single session)

Built for a runtime where **nothing survives the session**. That single fact drives
the whole design:

1. **Runs are ordered seed-major** — every model is trained at seed 0 before any
   model is trained at seed 1. If the session dies at 60%, you have one complete
   seed across the whole matrix (a usable table) rather than three seeds across
   an arbitrary 60% of it (not a table).
2. **`--time-budget` stops cleanly between runs** instead of being killed
   mid-training, where an interrupted run leaves nothing at all.
3. **Every result is printed to cell output as it completes.** Saved notebook
   output is the one thing you reliably keep, so nothing lives only in memory.
4. **Results are zipped and downloaded to your machine at the end** — and you can
   run that cell at any point mid-experiment.

**Setup:** Runtime → Change runtime type → **L4 GPU** (or A100), and enable
**Background execution** so the run survives closing the tab.

In [ ]:
import torch

assert torch.cuda.is_available(), \
    "No GPU. Runtime > Change runtime type > L4 GPU, then rerun this cell."

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}  |  {vram:.1f} GB")
print(f"torch {torch.__version__}")

In [ ]:
#@title Clone the repo and install dependencies
REPO_URL = "https://github.com/phoenixfin/aksantara-ocr.git"  #@param {type:"string"}

import os
from pathlib import Path

REPO = Path("/content/aksantara-ocr")
if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO}

os.chdir(REPO)
# torch/torchvision ship with Colab; reinstalling them is slow and occasionally
# pulls a version mismatched against the runtime's CUDA build.
!pip install -q timm pyyaml scikit-image tabulate
print(f"\nready: {Path.cwd()}")

## Pull the dataset from Mendeley Data

Results don't persist, but the *input* still has to arrive somehow each session.
Fetching straight from the DOI is the cleanest answer: one command, no Drive
mount, no re-uploading, and the notebook stays reproducible for anyone who reads
the paper.

**Published datasets only.** Mendeley's public API serves no drafts and no
embargoed datasets. If yours isn't published yet, use a fallback cell below.

In [ ]:
#@title Fetch from Mendeley Data
DOI = "10.17632/vfj32bpjsf.2"  #@param {type:"string"}

# Look before downloading: confirms the DOI resolves and shows the total size,
# so a typo costs a second rather than a long transfer.
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --list-only

In [ ]:
# Download to local disk and unpack the 13 per-script archives. Local disk
# matters: reading tens of thousands of small images off a mounted Drive is slow
# enough to become the bottleneck instead of the GPU.
#
# Each archive contains <Script>/<character>/<n>.png, so extracting all of them
# into one directory produces the layout the scanner expects directly.
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --out /content/raw

RAW_ROOT = "/content/raw"
!find {RAW_ROOT} -maxdepth 2 -type d | head -20

In [ ]:
#@title Fallbacks — uncomment one if the dataset isn't published yet

# Upload a zip from your machine:
# from google.colab import files
# up = files.upload()
# !mkdir -p /content/data && unzip -q -o "{list(up)[0]}" -d /content/data
# DATA_ROOT = "/content/data"

# Copy a zip from Drive:
# from google.colab import drive; drive.mount("/content/drive")
# !mkdir -p /content/data && unzip -q -o "/content/drive/MyDrive/aksara/dataset.zip" -d /content/data
# DATA_ROOT = "/content/data"

# Synthetic data, to rehearse the pipeline before the real data is ready:
# !python scripts/make_synthetic_data.py --out /content/data/synthetic
# DATA_ROOT = "/content/data/synthetic"
pass

In [ ]:
#@title Clean the dataset (v2 -> v3)
# The published v2 contains 3,109 duplicate images, 4 mislabelled copies,
# 4 stray photo-editor exports, and 6 misfiled Sunda folders. Every fix is
# declared in configs/cleaning_rules.yaml with its evidence, and the script
# writes a changelog rather than mutating the source.
#
# Running this from the v2 DOI keeps the whole pipeline reproducible for anyone
# reading the paper, whether or not a cleaned v3 has been published yet.
!python scripts/06_clean_dataset.py \
    --data-root "{RAW_ROOT}" --out /content/clean \
    --rules configs/cleaning_rules.yaml

CLEAN_ROOT = "/content/clean"

In [ ]:
#@title Pre-resize into a decode-cheap cache — do not skip this
CACHE_SIZE = 224  #@param {type:"number"}

# Source images run up to 1500x1500 and average ~9KB of PNG decode work each.
# Measured: 512px source -> 224px costs 6.65 ms/img per core, so Colab's two
# dataloader workers sustain only ~300 img/s and the GPU sits idle. A 224px
# cache drops that to 0.98 ms/img — 6.8x faster — moving the bottleneck back
# onto the GPU.
#
# Costs a few minutes once; saves hours across the matrix.
!python scripts/00b_build_cache.py \
    --data-root "{CLEAN_ROOT}" --out /content/data --size {CACHE_SIZE}

DATA_ROOT = "/content/data"

# Free the disk — neither the raw nor the cleaned full-size tree is needed again.
!rm -rf {RAW_ROOT} {CLEAN_ROOT}
!df -h /content | tail -1

In [ ]:
#@title Scan and build the shared splits
ARTIFACTS = "/content/artifacts"

# stratified, not grouped: filenames (1.png, 2.png, ...) carry no writer id, so
# writer-disjoint splitting is impossible. Reported accuracy is therefore an
# upper bound on generalization to unseen handwriting — state this in the paper.
!python scripts/01_prepare_data.py \
    --data-root "{DATA_ROOT}" \
    --out-dir "{ARTIFACTS}" \
    --split-strategy stratified

# Expected on cleaned v3: 97,383 images, 13 scripts, 889 classes, no warnings.
!python scripts/04_audit_dataset.py \
    --manifest "{ARTIFACTS}/manifest.csv" --out "{ARTIFACTS}/audit" | head -40

## Calibrate before committing the session

With no checkpointing, the matrix has to fit the session. Measure one run, then
multiply — don't guess. The next two cells give you a per-run cost and a count.

In [ ]:
#@title Time a couple of short runs
!python scripts/02_run_matrix.py --config configs/smoke_test.yaml \
    --artifacts "{ARTIFACTS}" --results "{ARTIFACTS}/results/smoke" 2>&1 | grep -E "run:|acc=|mean"

In [ ]:
#@title Count the real matrix, and project the cost
CONFIG = "configs/tier1_benchmark.yaml"  #@param ["configs/tier1_benchmark.yaml", "configs/tier1_ablation_aug.yaml", "configs/tier1_ablation_size.yaml", "configs/tier1_per_script.yaml", "configs/full_benchmark.yaml"]
MINUTES_PER_RUN = 8  #@param {type:"number"}
RESULTS = "/content/artifacts/results/tier1"

import subprocess, re

out = subprocess.run(
    ["python", "scripts/02_run_matrix.py", "--config", CONFIG,
     "--artifacts", ARTIFACTS, "--results", RESULTS, "--dry-run"],
    capture_output=True, text=True,
).stdout

n = int(re.search(r"expands to (\d+)", out).group(1))
hours = n * MINUTES_PER_RUN / 60
print(f"{n} runs x ~{MINUTES_PER_RUN}min = ~{hours:.1f}h")
print(f"seeds are run in order, so a partial run still yields complete single-seed tables")
if hours > 20:
    print("\nThis likely exceeds one session. Either lower `seeds` in the config,"
          "\ndrop models, or set TIME_BUDGET below and accept a partial matrix.")

In [ ]:
#@title Run the flat 895-class benchmark (optional)
TIME_BUDGET = 8.0  #@param {type:"number"}

# The layered pipeline below is the headline result; this flat 895-way run is
# the comparison baseline showing what the hierarchy buys. Skip it if the
# session is tight.
#
# Set TIME_BUDGET a couple of hours under the session limit you expect: the
# runner predicts whether the next run fits and stops cleanly if it does not,
# so you always end on a completed run with results written and printed.
!python scripts/02_run_matrix.py --config "{CONFIG}" \
    --artifacts "{ARTIFACTS}" --results "{RESULTS}" \
    --time-budget {TIME_BUDGET}

In [ ]:
#@title Layered classifier: both stages, then end-to-end evaluation
# Stage 1 (script_id) and stage 2 (per_script) must share a results dir and
# seeds — 05_hierarchical_eval pairs them by seed so the reported system is one
# coherent pipeline rather than a mix across runs.
!python scripts/02_run_matrix.py --config configs/layered.yaml \
    --artifacts "{ARTIFACTS}" --results "{RESULTS}" --time-budget 6

# End-to-end accuracy, decomposed into routing loss vs character loss.
# Exactly computable: labels are script-qualified, so a misrouted image can
# never yield the correct label.
!python scripts/05_hierarchical_eval.py --results "{RESULTS}"

# CPU-only — costs no GPU time, and anchors the deep results against a
# non-deep baseline.
!python scripts/03_run_classical.py --artifacts "{ARTIFACTS}"

In [ ]:
#@title Remaining tier-1 configs (run if the budget allows)
# All write into the same RESULTS dir, so the report aggregates them together.
for cfg in [
    "configs/tier1_ablation_aug.yaml",
    "configs/tier1_ablation_size.yaml",
    "configs/tier1_per_script.yaml",
]:
    print(f"\n{'=' * 70}\n{cfg}\n{'=' * 70}")
    !python scripts/02_run_matrix.py --config {cfg} \
        --artifacts "{ARTIFACTS}" --results "{RESULTS}" --time-budget 4

# CPU-only — costs no GPU time, and anchors the deep results against a
# non-deep baseline.
!python scripts/03_run_classical.py --artifacts "{ARTIFACTS}"

In [ ]:
#@title Print the tables (this output is saved with the notebook)
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 400)
report = Path(RESULTS) / "report"

for name in ["main_benchmark", "ablation_augmentation", "ablation_image_size",
             "ablation_pretrained", "per_script"]:
    path = report / f"{name}.csv"
    if path.exists():
        print(f"\n{'=' * 70}\n{name}\n{'=' * 70}")
        print(pd.read_csv(path).to_string(index=False))

# Print the raw per-run table too — if the zip download fails, this output is
# still enough to rebuild every table by hand.
raw = report / "raw_results.csv"
if raw.exists():
    print(f"\n{'=' * 70}\nraw results (every run)\n{'=' * 70}")
    print(pd.read_csv(raw).to_string(index=False))

failures = Path(RESULTS) / "failures.jsonl"
if failures.exists():
    print(f"\nSome runs failed:\n{failures.read_text()[:3000]}")

In [ ]:
#@title Download everything to your machine
# Safe to run at any point, including mid-experiment from a second browser tab.
# Nothing here survives the session, so run it before you walk away.
from google.colab import files

!cd /content && zip -qr aksara_results.zip artifacts \
    -x "artifacts/results/*/*/test_predictions.npz"
!du -h /content/aksara_results.zip

# test_predictions.npz is excluded above because raw logits dominate the archive
# size. Drop the -x flag if you want them for post-hoc analysis.
files.download("/content/aksara_results.zip")

## If the session dies partway

You lose the run outputs, but not the experiment. Because ordering is
seed-major, whatever completed is a coherent slice: seed 0 across every model,
then seed 1, and so on. Concretely:

- **Saved notebook output** still contains every completed run's accuracy and
  macro-F1, printed as it finished.
- To continue, edit the config's `seeds:` to only the ones you still need
  (`seeds: [1, 2]`) and rerun. Splits are deterministic from the same seed, so
  a later session reproduces byte-identical splits and the results remain
  directly comparable.
- If you can tolerate mounting Drive for output, pointing `--artifacts` at
  `/content/drive/MyDrive/...` restores full resume — completed runs are then
  detected and skipped automatically.